In [1]:
import os

In [2]:
os.chdir('..')

In [3]:
%pwd

'd:\\Chicken Disease Classification'

In [4]:
from dataclasses import dataclass
from pathlib import Path

In [5]:
@dataclass
class PrepareCallbackConfig:
    root_dir: Path
    tensorboard_root_log_dir: Path
    checkpoint_model_filepath: Path

In [6]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories

In [14]:
class ConfigurationManager:
    def __init__(self,
                 config_filepath=CONFIG_FILE_PATH,
                 params_filepath=PARAMS_FILE_PATH):
        self.config=read_yaml(config_filepath)
        self.params=read_yaml(params_filepath)
        
        create_directories([self.config.artifacts_root])
    
    def get_prepare_callback_config(self)->PrepareCallbackConfig:
        config=self.config.prepare_callbacks
        model_ckpt_dir=os.path.dirname(config.checkpoint_model_filepath)
        create_directories([Path(config.tensorboard_root_log_dir),Path(model_ckpt_dir)])
        prepare_callback_config=PrepareCallbackConfig(
            root_dir=Path(config.root_dir),
            tensorboard_root_log_dir= Path(config.tensorboard_root_log_dir),
            checkpoint_model_filepath=Path(config.checkpoint_model_filepath)
        )
        return prepare_callback_config

In [8]:
import urllib.request as request
from zipfile import ZipFile
import time
import tensorflow as tf

In [23]:
class PrepareCallbacks:
    def __init__(self,config:PrepareCallbackConfig):
        self.config=config
    
    @property
    def _create_tb_callbacks(self):
        timestamp=time.strftime("%Y-%m-%d-%H-%M-%S")
        tb_running_log_dir=os.path.join(self.config.tensorboard_root_log_dir,f"tb_logs_at_{timestamp}")
        return tf.keras.callbacks.TensorBoard(log_dir=tb_running_log_dir)
    
    @property
    def _create_ckpt_callbacks(self):
        return tf.keras.callbacks.ModelCheckpoint(filepath=self.config.checkpoint_model_filepath,
                                                  save_best_only=True)
    
    def get_tb_ckpt_callbacks(self):
        return [self._create_tb_callbacks,
                self._create_ckpt_callbacks]
        

In [24]:
try:
    config=ConfigurationManager()
    prepare_callback_config=config.get_prepare_callback_config()
    prepare_callback=PrepareCallbacks(config=prepare_callback_config)
    callback_list=prepare_callback.get_tb_ckpt_callbacks()
except Exception as e:
    raise e

[2026-03-24 00:22:42,212: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-03-24 00:22:42,225: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-24 00:22:42,229: INFO: common: created directory at: artifacts]
[2026-03-24 00:22:42,229: INFO: common: created directory at: artifacts\prepare_callbacks\tensorboard_log_dir]
[2026-03-24 00:22:42,229: INFO: common: created directory at: artifacts\prepare_callbacks\checkpoint_dir]
